### Medical Appointment Data Cleaning Pipeline

### Import Functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/my_files/")

### Data ingestion and schema

In [0]:
df = spark.read.format('csv').option("inferSchema", True).option("header", True).load("/Volumes/workspace/default/my_files/KaggleV2-May-2016.csv")

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
df_schema = StructType([
    StructField("PatientID", LongType()),
    StructField("AppointmentID", IntegerType()),
    StructField("Gender", StringType()),
    StructField("ScheduledDay", TimestampType()),
    StructField("AppointmentDay", TimestampType()),
    StructField("Age", IntegerType()),
    StructField("Neighbourhood", StringType()),
    StructField("Scholarship", IntegerType()),
    StructField("Hipertension", IntegerType()),
    StructField("Diabetes", IntegerType()),
    StructField("Alcoholism", IntegerType()),
    StructField("Handcap", IntegerType()),
    StructField("SMS_received", IntegerType()),
    StructField("No-show", StringType()),
])

In [0]:
df_custom_schema = spark.read.format('csv').schema(df_schema).option("header", True).load("/Volumes/workspace/default/my_files/KaggleV2-May-2016.csv")

In [0]:
print("INFERRED SCHEMA")
df.printSchema()

print("CUSTOM SCHEMA")
df_custom_schema.printSchema()

### Selecting and renaming columns

In [0]:
df = df.withColumnRenamed("PatientId", "patient_id") \
       .withColumnRenamed("AppointmentID", "appointment_id") \
       .withColumnRenamed("Gender", "gender") \
       .withColumnRenamed("ScheduledDay", "scheduled_day") \
       .withColumnRenamed("AppointmentDay", "appointment_day") \
       .withColumnRenamed("Age", "age") \
       .withColumnRenamed("Neighbourhood", "neighbourhood") \
       .withColumnRenamed("Scholarship", "scholarship") \
       .withColumnRenamed("Hipertension", "hypertension") \
       .withColumnRenamed("Diabetes", "diabetes") \
       .withColumnRenamed("Alcoholism", "alcoholism") \
       .withColumnRenamed("Handcap", "handicap") \
       .withColumnRenamed("SMS_received", "sms_received") \
       .withColumnRenamed("No-show", "no_show")

In [0]:
df_alias = df.select(col("patient_id").alias("patient_identifier"))

In [0]:
df_alias.display()

### Filtering and investigating records

#### Scenario 1

In [0]:
df.filter(col("age") < 0).display()

#### Scenario 2


In [0]:
df.filter(col("age") < 18).display()

In [0]:
df.filter(col("no_show")=="Yes").display()

In [0]:
df.filter((col("sms_received") == 1) & (col("no_show")=="Yes")).display()

In [0]:
df.filter((col("scholarship") == 1) & (col("no_show")=="Yes")).display()

In [0]:
df.filter((col("hypertension") == 1) & (col('diabetes') == 1) & (col("no_show")=="Yes")).display()

In [0]:
df.filter((col("age").isNull()) | col("neighbourhood").isNull()).display()

### Creating and modifying columns

In [0]:
df = df.withColumn("data_source", lit("Kaggle Medical Dataset"))
df.display()

In [0]:
df = df.withColumn("appointment_status",when(col("no_show") == "Yes", "No-show").otherwise("Show"))
df.display()

In [0]:
df = df.withColumn("gender", upper(col("gender")))
df.display()

In [0]:
df = df.withColumn("age_group", when((col("age")).isNull(), "Unknown").when((col("age") ) < 0, "Invalid_age").when((col("age")) <18, "Under 18").when((col("age")) <36, "18-35").when((col("age")) <51, "36-50").when((col("age")) <66, "51-65").otherwise("Above 65"))
df.display() 

In [0]:
df = df.withColumnRenamed("no_show", "no_show_flag")
df.display()


### Type casting and sorting

In [0]:
df.select(
    "scheduled_day",
    "appointment_day"
).show(10, truncate=False)

In [0]:
df = df.withColumn('scheduled_day', col('scheduled_day').cast(TimestampType()))
df.display()

In [0]:

df = df.withColumn('appointment_day', col('appointment_day').cast(TimestampType()))
df.display()

In [0]:
df = df.withColumn('age', col('age').cast(IntegerType()))
df.display()

In [0]:
df = df.withColumn('hypertension', col('hypertension').cast(IntegerType()))
df = df.withColumn('diabetes', col('diabetes').cast(IntegerType()))
df = df.withColumn('alcoholism', col('alcoholism').cast(IntegerType()))
df = df.withColumn('handicap', col('handicap').cast(IntegerType()))
df.display()

In [0]:
df.sort(col("age").desc()).display()

In [0]:
df.sort(["appointment_day", "age"], ascending = [1, 0]).display()

In [0]:
df.limit(20).display()

In [0]:
df.limit(100).display()

### Removing columns and duplicates


In [0]:
df.display()

In [0]:
df.groupBy(*df.columns).count().filter(col("count")>1).display()

In [0]:
df = df.dropDuplicates()
df.display()

In [0]:
df.groupBy("appointment_id").count().filter(col("count")>1).display()

In [0]:
df.display()

In [0]:
df.groupBy("patient_id").count().filter(col("count")>1).display()

In [0]:
indicator_columns = [
    "scholarship",
    "hypertension",
    "diabetes",
    "alcoholism",
    "handicap",
    "sms_received"
]

for column in indicator_columns:
    df.filter(
        ~col(column).isin(0, 1) & col(column).isNotNull()
    ).select(column).distinct().display()

In [0]:
df_clean = df.dropDuplicates()

In [0]:
print("Final schema:")
df_clean.printSchema()

print("Final row count:")
print(df_clean.count())

df_clean.limit(10).display()

### Data quality decisions

In [0]:
df_clean = df_clean.withColumn(
    "appointment_day",
    to_date(col("appointment_day"))
).withColumn(
    "scheduled_day",
    to_date(col("scheduled_day"))
)

In [0]:
appointments_before_scheduled = df_clean.filter(col("appointment_day") < col("scheduled_day"))
appointments_before_scheduled.display()

In [0]:
null_dates = df_clean.filter(~col("appointment_day").isNotNull() | ~col("scheduled_day").isNotNull())
null_dates.display()

In [0]:
df_clean.display()

In [0]:
same_day_appointments = df_clean.filter(col("appointment_day").isNotNull() & col("scheduled_day").isNotNull() & (col("appointment_day") == col("scheduled_day")))
same_day_appointments.display()

### Investigate Age column

In [0]:
df_clean.filter(col("age") < 0).display()

In [0]:
df_clean.filter(col("age") > 100).display()

In [0]:
df_clean.filter(col("age").isNull()).display()

In [0]:
df_clean.filter(col("age") == 0).display()

### Age statistics

In [0]:
df_clean.select(
    min("age").alias("minimum_age"),
    max("age").alias("maximum_age")
).display()

In [0]:
df_clean.groupBy("age").count().orderBy("age").display()

In [0]:
print("Negative ages:", df_clean.filter(col("age") < 0).count())

print("Ages above 100:", df_clean.filter(col("age") > 100).count())

print("Null ages:", df_clean.filter(col("age").isNull()).count())

print("Age zero:", df_clean.filter(col("age") == 0).count())

In [0]:
df_clean = df_clean.withColumn(
    "age_validation_status",
    when(col("age") < 0, "invalid_age")
    .when(col("age") > 100, "review_age")
    .otherwise("valid")
)

In [0]:
df_clean.groupBy("age_validation_status").count().display()

In [0]:
df_clean.filter(col("age_validation_status") != "valid").display()

In [0]:
indicator_columns = [
    "scholarship",
    "hypertension",
    "diabetes",
    "alcoholism",
    "handicap",
    "sms_received"
]

In [0]:
for column in indicator_columns:
    invalid_count = df_clean.filter(col(column).isNotNull() & (~(col(column).isin(0, 1)))).count()
    nulll_count = df_clean.filter(col(column).isNull()).count()
    distinct_value = df_clean.filter(col(column).isNotNull() & (~(col(column).isin(0, 1)))).select(column).distinct()
    print(f"Column {column} has {invalid_count} invalid values, {nulll_count} null values, and {distinct_value.count()} distinct values.")
    distinct_value.display()

In [0]:
df_clean.groupBy("appointment_id") \
    .count() \
    .filter(col("count") > 1) \
    .orderBy(col("count").desc()) \
    .display()

In [0]:
df_clean.groupBy("gender").count().display()

In [0]:
df_clean.filter(~(col("gender").isin("F", "M"))).display()
df_clean.filter(col("neighbourhood").isNull()).display
df_clean.filter(~(col("no_show").isin("Yes", "No"))).display()
df_clean.filter((col("patient_id").isNull()) | (col("appointment_id").isNull())).display()
df_clean.filter(col("neighbourhood").isNull() | (trim(col("neighbourhood")) == "")).display()

### Investigate gender

In [0]:
df_clean.groupBy('gender').count().display()

In [0]:
df_clean.filter(col("gender").isNull()).display()
df_clean.filter(col("gender").isNull() & (trim(col("gender") == ""))).display()

In [0]:
df_clean.filter(col("gender").isNotNull() & ~(trim(col("gender").isin("F", "M")))).display()

### Investigate no_show_flag

In [0]:
df_clean.groupBy('no_show_flag').count().display()

In [0]:
df_clean.filter(col("no_show_flag").isNull()).display()
df_clean.filter(col("no_show_flag").isNull() | (trim(col("no_show_flag") == ""))).display()

In [0]:
df_clean.filter(col("no_show_flag").isNotNull() & ~(trim(col("no_show_flag").isin("Yes", "No")))).display()

### Investigate neighbourhood

In [0]:
df_clean.groupBy('neighbourhood').count().display()

In [0]:
df_clean.filter(col("neighbourhood").isNull()).display()
df_clean.filter(col("neighbourhood").isNull() & (trim(col("neighbourhood") == ""))).display()

In [0]:
df_clean.groupBy("neighbourhood").count().orderBy(col("count").desc()).limit(10).display()

### investigate the identifiers

In [0]:
df_clean.filter(col("patient_id").isNull()).count()

In [0]:
df_clean.filter(col("appointment_id").isNull()).count()

In [0]:
df_clean.filter(trim(col("neighbourhood")) == "").count()

#### Check appointment status consistency

In [0]:
df_clean.filter((col("no_show_flag") == "Yes") & (col("appointment_status") != "No-show")).count()

In [0]:
df_clean.filter((col("no_show_flag") == "No") & (col("appointment_status") != "Show")).count()

In [0]:
df_clean.display()

#### Validate age_group

In [0]:
df_clean.filter(col('age') < 0).count()

In [0]:
df_clean.filter(
    (col("age") == 0) &
    (col("age_group") != "Under 18")
).count()

In [0]:
df_clean.filter((col('age') > 100) & (col('age_group') != 'Above 65')).count()

In [0]:
df_clean.filter(col('age').isNull() & (col('age_group').isNull() | (col('age_group') != 'Unknown'))).count()

In [0]:
df_clean.filter(
    col("age") < 0
).select(
    "patient_id",
    "appointment_id",
    "age",
    "age_group",
    "age_validation_status"
).display()

In [0]:
df_clean.filter(
    (col("age") > 100) &
    (col("age_group") != "Above 65")
).count()

In [0]:
df_clean.filter(
    col("age").isNull() &
    (col("age_group") != "Unknown")
).count()

In [0]:
boundary_ages = [18, 36, 51, 66]

for age_value in boundary_ages:
    df_clean.filter(col("age") == age_value).select(
        "age", "age_group"
    ).distinct().display()

In [0]:
df_clean.filter(
    ((col("age") < 0) & (col("age_validation_status") != "invalid_age")) |
    ((col("age") > 100) & (col("age_validation_status") != "review_age")) |
    ((col("age") >= 0) & (col("age") <= 100) &
     (col("age_validation_status") != "valid"))
).select(
    "age", "age_group", "age_validation_status"
).display()

### Build a data quality summary

In [0]:
# 1. Duplicate records
total_records = df_clean.count()
unique_records = df_clean.dropDuplicates().count()
duplicate_count = total_records - unique_records

# 2. Age checks
invalid_ages = df_clean.filter(col('age') < 0).count()
ages_requiring_review = df_clean.filter(col('age') > 100).count()
zero_ages = df_clean.filter(col('age') == 0).count()

# 3. Records where either date is missing
null_dates = df_clean.filter(
    col('scheduled_day').isNull() |
    col('appointment_day').isNull()
).count()

# 4. Appointment before scheduling
appointments_before_scheduling = df_clean.filter(
    col('scheduled_day').isNotNull() &
    col('appointment_day').isNotNull() &
    (col('appointment_day') < col('scheduled_day'))
).count()

# 5. Invalid indicator values across six columns
indicator_columns = [
    'scholarship',
    'hypertension',
    'diabetes',
    'alcoholism',
    'handcap',
    'sms_received'
]

invalid_indicator_count = 0
for c in indicator_columns:
    invalid_indicator_count += df_clean.filter(
        col(c).isNotNull() &
        (~col(c).isin(0, 1))
    ).count()

# 6. Invalid categorical values
# Count non-null values that are not in the expected categories.
invalid_gender = df_clean.filter(
    col('gender').isNotNull() &
    (~trim(col('gender')).isin('M', 'F'))
).count()

invalid_no_show = df_clean.filter(
    col('no_show_flag').isNotNull() &
    (~trim(col('no_show_flag')).isin('Yes', 'No'))
).count()

# Count null or blank neighbourhood values.
invalid_neighbourhood = df_clean.filter(
    col('neighbourhood').isNull() |
    (trim(col('neighbourhood')) == '')
).count()

invalid_categorical_count = (
    invalid_gender +
    invalid_no_show +
    invalid_neighbourhood
)

# Build the summary
summary_data = [
    ('Duplicate records', duplicate_count),
    ('Invalid ages (age < 0)', invalid_ages),
    ('Ages requiring review (age > 100)', ages_requiring_review),
    ('Zero ages (age == 0)', zero_ages),
    ('Null dates', null_dates),
    ('Appointment before scheduling', appointments_before_scheduling),
    ('Invalid indicator values', invalid_indicator_count),
    ('Invalid categorical values', invalid_categorical_count)
]

quality_summary = spark.createDataFrame(
    summary_data,
    ['Check', 'Count']
)

quality_summary.display()

In [0]:
quality_results = {
    row["Check"]: row["Count"]
    for row in quality_summary.collect()
}

for check_name, issue_count in quality_results.items():
    print(f"{check_name}: {issue_count}")